# MERIT-UNETR++ training on Colab

Select a **GPU** Colab kernel in VS Code, then run cells in order. This uses fold 0, 11 index slices, 256x256, batch size 1 and native scaled intensity. The 256 GPU profile passed on one T4 with synthetic inputs; the real-CT pilot below is required before the training cell.

The three cases with missing spatial units are explicitly assumed to be mm. This does not verify HU, and the proposed fitted normalization is not implemented. Current grouping is volume-level, not independently verified patient-level. The test partition is not evaluated by this notebook.

In [ ]:
from pathlib import Path
import hashlib, json, os, re, subprocess, sys, tempfile

REPO_URL = 'https://github.com/torusama/merit-covid-ct.git'
BRANCH = 'train/merit-unetrpp-colab'  # Publish this branch before cloning it.
PROJECT = Path('/content/merit-covid-train')
DATA = Path('/content/drive/MyDrive/PROJECT HCMUS/Paper/merit-covid-colab-2026-11-08/data/covid19_20')
RUN_ROOT = Path('/content/drive/MyDrive/PROJECT HCMUS/Paper/merit_runs')
RUN_NAME = 'fold0_256_native_v1'  # Use a new name for a different experiment.
FOLD = 0
EPOCHS = 1  # Total epoch budget; increase to continue a successful run.
CONFIG = PROJECT / 'configs/merit_colab_256.yaml'

def run_command(args, cwd=None):
    process = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    if code:
        raise subprocess.CalledProcessError(code, args)

## 1. Get the training branch
The branch and its commits must be pushed to GitHub first. The notebook refuses to update a dirty checkout or one belonging to a different branch/repository. Use a new run name for a changed code revision; do not update code in the middle of a run.

In [ ]:
if not PROJECT.exists():
    run_command(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT)])
else:
    if not (PROJECT / '.git').is_dir():
        raise RuntimeError(f'{PROJECT} exists but is not a Git checkout')
    def git_output(*args):
        return subprocess.check_output(['git', *args], cwd=PROJECT, text=True).strip()
    if git_output('remote', 'get-url', 'origin') != REPO_URL or git_output('branch', '--show-current') != BRANCH:
        raise RuntimeError('Existing checkout has a different origin or branch')
    if git_output('status', '--porcelain'):
        raise RuntimeError('Existing checkout has changes; preserve them before updating')
    run_command(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=PROJECT)
os.chdir(PROJECT)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
print('Git commit:', commit)
if not CONFIG.is_file():
    raise FileNotFoundError(f'Training config missing: {CONFIG}')

## 2. Mount Drive, check all files and connect the saved split
The dataset is read through the Drive mount. This checks manifest paths, not clinical identity or file checksums.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
manifest_path = DATA / 'cases.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'Manifest not found: {manifest_path}')
records = json.loads(manifest_path.read_text(encoding='utf-8-sig'))
if isinstance(records, dict):
    records = records['cases']
missing = [str((DATA / r[k]).relative_to(DATA)) for r in records
           for k in ('image', 'mask' if 'mask' in r else 'infection_mask') if not (DATA / r[k]).is_file()]
print(f'Cases: {len(records)}, missing CT/mask files: {len(missing)}')
if len(records) != 199 or missing:
    print('Missing:', *missing, sep='\n  ')
    raise RuntimeError('Complete the dataset upload before continuing')
link = PROJECT / 'data/covid19_20'
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink():
    if link.resolve() != DATA.resolve():
        raise RuntimeError('Existing data link points elsewhere')
elif link.exists():
    raise RuntimeError(f'Existing dataset path will not be overwritten: {link}')
else:
    link.symlink_to(DATA, target_is_directory=True)
print('Dataset:', link.resolve())

In [ ]:
run_command([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[unetrpp]'], cwd=PROJECT)
run_command([sys.executable, 'scripts/merit_pipeline.py', 'check-split', '--config', str(CONFIG)], cwd=PROJECT)
run_command([sys.executable, '-c', "import torch; print('CUDA:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'); assert torch.cuda.is_available(), 'Select a GPU Colab kernel'"], cwd=PROJECT)

## 3. Required real-CT pilot
Four actual training steps on fold-0 TRAIN case A-0025, then checkpoint reload and complete native-grid inference. Artifacts in `/content` are temporary and diagnostic. The training cell requires this pilot to pass in the current runtime.

In [ ]:
pilot_report = None
pilot_gate = None
pilot_output = Path(tempfile.mkdtemp(prefix='merit_pilot_')) / 'run'
run_command([sys.executable, '-u', 'scripts/merit_preflight.py', 'pilot', '--config', str(CONFIG),
    '--fold', str(FOLD), '--case-id', 'volume-covid19-A-0025', '--size', '256',
    '--steps', '4', '--device', 'cuda', '--output', str(pilot_output)], cwd=PROJECT)
pilot_report = json.loads((pilot_output / 'pilot.json').read_text(encoding='utf-8'))
if pilot_report.get('status') != 'PASS':
    raise RuntimeError(f'Pilot failed: {pilot_report.get("error", "unknown failure")}')
pilot_gate = (commit, hashlib.sha256(CONFIG.read_bytes()).hexdigest(), FOLD, str(DATA.resolve()))
print('Real CT pilot: PASS; output:', pilot_output)

## 4. Train / resume one fold
Run this cell when ready. The initial budget is one **full** training epoch plus full validation, which may take hours. Set `EPOCHS` to a larger total to continue. Each completed epoch saves `last.pt` and, when validation improves, `best.pt` under `RUN_ROOT/RUN_NAME` on Drive. Interrupted epochs restart from the last completed checkpoint. Changing the commit, config, dataset path or fold requires a new run name.

In [ ]:
if not isinstance(globals().get('pilot_report'), dict) or pilot_report.get('status') != 'PASS':
    raise RuntimeError('Run the real-CT pilot successfully before training')
current_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=PROJECT, text=True).strip()
if dirty or current_commit != commit:
    raise RuntimeError('Code changed after setup; use a clean checkout and rerun the pilot')
expected_gate = (current_commit, hashlib.sha256(CONFIG.read_bytes()).hexdigest(), FOLD, str(DATA.resolve()))
if globals().get('pilot_gate') != expected_gate:
    raise RuntimeError('Configuration, fold or data path changed after the pilot; rerun it')
if type(EPOCHS) is not int or EPOCHS < 1 or FOLD not in range(5):
    raise ValueError('EPOCHS must be positive and FOLD must be 0..4')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', RUN_NAME) or RUN_NAME in ('.', '..'):
    raise ValueError('Choose a simple run name')
run_dir = RUN_ROOT / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
record_path = run_dir / 'run_provenance.json'
record = {'git_commit': commit, 'config_sha256': hashlib.sha256(CONFIG.read_bytes()).hexdigest(),
          'dataset': str(DATA.resolve()), 'fold': FOLD, 'training_order': 'case_grouped_v1'}
if record_path.exists():
    if json.loads(record_path.read_text(encoding='utf-8')) != record:
        raise RuntimeError('Existing run has different code, config, data path or fold; choose a new RUN_NAME')
else:
    if any(run_dir.iterdir()):
        raise RuntimeError('Output directory is not empty; choose a new RUN_NAME')
    with record_path.open('x', encoding='utf-8') as handle:
        json.dump(record, handle, indent=2)
checkpoint = run_dir / 'last.pt'
args = [sys.executable, '-u', 'scripts/train_colab.py', '--config', str(CONFIG),
        '--fold', str(FOLD), '--output', str(run_dir), '--epochs', str(EPOCHS)]
if checkpoint.is_file():
    print('Resuming:', checkpoint, '(increase EPOCHS beyond completed epochs)')
    args += ['--resume', str(checkpoint)]
elif any(p.name != record_path.name for p in run_dir.iterdir()):
    raise RuntimeError('Run contains files but has no last.pt; inspect it or choose a new RUN_NAME')
print('Training output:', run_dir)
run_command(args, cwd=PROJECT)

Model checkpoints are research artifacts, not clinically validated results. Do not use internal or external test results to choose input size, normalization, threshold or checkpoint.